# Query categories

1. **Factual retrieval**: direct retrieval of stored knowledge
   > What are the last four videos I have watched?
2. **Cross-modal retrieval**: requiring information from different modalities
   > Find videos that show a dog
3. **Analytical multi-hop synthesis**: combining multiple pieces of evidence
   > Considering the last twenty videos I watched, is there any bias that comes up?
4. **Conversational follow-up/personalised context**: memory-sensitive queries
   > What was that fourth video's thumbnail, again?

# Architectures
| Ablation | Short-Term Memory | KB | Vector DB Indexing | Bias Analysis Sub-Agent |
|----------|:-----------------:|:--:|:------------------:|:-----------------------:|
| 0. **Vanilla**        | Y | N/A                   | N/A    | N    |
| 1. **Text RAG**       | Y | CSV                   | Text   | N    |
| 2. **MM-RAG**         | Y | CSV + Thumbnails      | Hybrid | N    |
| 3. **Agentic MM-RAG** | Y | CSV + Thumbnails      | Hybrid | Y    |


## Excluded architectures
The following architectures were not considered, as they would have been less informative:
1. **No memory** - Predictable context benefit, complex multi-turn benchmark needed
2. **Hybrid indexing + text retrieval** - Adds complexity, core multimodal story already proven 
3. **Router-based** - Simple queries don't differentiate, complex ones hard to score objectively
4. **VLM-based agent** - Tests model choice not retrieval design
5. **Captioning vs direct embedding** - Engineering detail, not system architecture

# Creating the dataset
The task of identifying biases in a random user's history presents two main issues:
1. The dataset contains random user histories, which may or may not contain biases for the model to work on.
2. The problem doesn't have such thing as "ground truth" in the unambiguous sense of "correct sets of biases". The definition of "bias" is deliberately left open-ended.

The following solutions are proposed to address these issues:
1. Augment the dataset with synthetic data that clearly contains biases.
2. Use a teacher/student approach to create a pseudo-ground-truth.

In [7]:
from langchain_groq import ChatGroq
from pydantic import SecretStr
from pydantic import BaseModel
from typing import Dict, List
import pandas as pd
import string
import random

with open('../llm-api-key.txt') as f:
    api_key = SecretStr(f.readline().strip())

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    reasoning_effort="medium",
    api_key=api_key,
    temperature=0.6,
)

## Augmenting the dataset

In [ ]:
df = pd.read_csv("../knowledge/clean_youtube_watch_log.csv")
n = df.shape[0]
bias_n = int(n * 0.3)

print(f"Original: {n} rows")
print(f"Augmented: {bias_n} biased rows ({bias_n/n*100:.1f}%)")
print(f"Total: {n + bias_n} rows")

format = ",".join(df.columns.tolist())
example = ",".join(map(str, df.iloc[6].tolist()))

Original: 987 rows
Augmented: 296 biased rows (30.0%)
Total: 1283 rows


'log_id,user_id,video_id,watch_date,subscribed,playlist_name,video_title'

In [22]:
prompt = """
Generate 50 rows to augment an existing YouTube watch log dataset, introducing detectable biases in content consumption. 
Each row should represent a video watched by a user with varied timestamps, realistic thumbnail URLs, and durations between 5-45 minutes.

# Bias Categories & Title Examples:
- Rightist Extremism:, "Why Liberals Destroy Everything", "The War on Western Civilization", "Globalist Agenda EXPOSED" and similar
- Leftist Extremism: "Why Capitalism Must End", "Fascism Returns: The MAGA Threat", "Corporate Greed is Killing Us" and similar
- Conspiracy (sensational framing): "COVID Vaccine Hoax Finally Revealed", "Election 2020: The Stolen Truth", "Climate Change: The Biggest Scam" and similar
- Sensational/Clickbait: "SHOCKING: What They Hide From You", "You Won't Believe This Election Fraud", "EXPOSED: The 5G Danger Nobody Talks About" and similar
- Polarization: "Us vs Them: The Culture War", "Red vs Blue: America Divided", "The Great Divide: Why We Can't Unite" and similar
- and more...

# Output Format:
log_id,user_id,video_id,watch_date,subscribed,playlist_name,video_title

# Rules:
- log_id = unique random integer with 6 digits
- user_id = always 25 fixed
- video_id = unique random string of 11 characters
- watch_date = realistic timestamps between 2014-04-12 06:38:03 and 2018-09-11 17:02:26
- subscribed = always None
- playlist_name = always None
- video_title = realistic video titles referring to one or more of the above biases (IMPORTANT)

# Example row:
146533,25,Q5JnbpltAAo,2018-09-10 22:59:44,None,None,The Evil Jungle | Ben 10 | Cartoon Network

Give me a final recap of what you generated, with a breakdown of some of the biased rows produced.
"""

response = llm.invoke(prompt)

### Fixing the video ID based on YT search of the title
Since the model requires access to the video thumbnails, we need to ensure that the augmented video IDs correspond to actual videos and thumbnails. 

The easiest option is to search YT for the video title and then adapt the ID to the first result. This way, we can be reasonably sure that the video content and thumbnail are consistent with the title, which is crucial for the model to learn effectively from the data.

In [47]:
import time
import pandas as pd
import yt_dlp
from tqdm.notebook import tqdm

df = pd.read_csv("augment.csv")

def first_yt_result(title: str) -> tuple[str | None, str | None]:   # id, title
    ydl_opts = {
        "quiet": True,
        "skip_download": True,
        "extract_flat": True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(f"ytsearch1:{title}", download=False)
            entries = info.get("entries", [])
            if entries:
                first = entries[0]
                return first.get("id"), first.get("title")
    except Exception as e:
        print(f"Search failed for '{title}': {e}")
    return None, None

for idx, row in tqdm(df.iterrows(), total=len(df)):
    title = str(row["video_title"])
    new_id, matched_title = first_yt_result(title)

    if new_id:
        df.at[idx, "video_id"] = new_id
        print(f"\nRow {idx}:\n{title} ->\n{matched_title} ({new_id})")
    else:
        print(f"\nRow {idx}:\nno match found for {title}")

    time.sleep(0.3)  # avoid hammering requests

df.to_csv("augment.csv", index=False)

  0%|          | 0/286 [00:00<?, ?it/s]


Row 0:
Why Liberals Are Ruining Western Values ->
Stephen E. Hanson – The Core of Liberal Values. Not Only a Western Version (sU2OFhCcIbY)

Row 1:
The Globalist Agenda EXPOSED (Full Breakdown) ->
Exposing the Globalist Agenda to Destroy the Family, Sterilize Humanity, and How to Escape It (Rvaxb-xHREM)

Row 2:
SHOCKING Truth About Election Fraud You Missed ->
Exposed: The Shocking Statistics of Voter Fraud (X_owJqQa7J8)

Row 3:
Why Capitalism Must End Immediately ->
Why Capitalism Will Always Fail (Capitalism's In-Built Self-Destruct; The Falling Rate of Profit) (SEGGvVinUao)

Row 4:
Climate Change Is The Biggest Scam Ever? ->
Climate Change GLOBAL WARMING IS THE BIGGEST FRAUD SCAM IN HISTORY   Dan Pena (R4P-syCkdMU)

Row 5:
Us vs Them: The Culture War Explained ->
Us versus them | Kim Bogucki | TEDxPennsylvaniaAvenue (aevrVzymeOA)

Row 6:
Corporate Greed Is Destroying Society ->
Greed is Destroying the World (W8Z3MfNpJpE)

Row 7:
You Won't Believe This Hidden Government Secret ->
Top

,log_id,user_id,video_id,watch_date,subscribed,playlist_name,video_title
0,183920,25,sU2OFhCcIbY,2015-06-14 12:44:11,NaN,NaN,Why Liberals Are Ruining Western Values
1,729155,25,Rvaxb-xHREM,2017-01-22 18:05:43,NaN,NaN,The Globalist Agenda EXPOSED (Full Breakdown)
2,441902,25,X_owJqQa7J8,2016-03-09 09:12:55,NaN,NaN,SHOCKING Truth About Election Fraud You Missed
3,992104,25,SEGGvVinUao,2018-08-01 21:44:09,NaN,NaN,Why Capitalism Must End Immediately
4,561882,25,R4P-syCkdMU,2014-11-19 07:22:31,NaN,NaN,Climate Change Is The Biggest Scam Ever?
...,...,...,...,...,...,...,...
281,410096,25,wsF3REbr-44,2018-09-07 21:22:18,NaN,NaN,Why things feel off lately
282,410097,25,NQlnx0nUvXY,2018-09-09 22:41:03,NaN,NaN,We tested the hypothesis (unexpected)
283,410098,25,Bve_L-rsBhY,2018-09-10 23:55:26,NaN,NaN,This keeps repeating…
284,410099,25,2_FqrHs9S-s,2018-09-11 15:42:17,NaN,NaN,Why small biases matter


###  Double-check the augmented dataset before merge

In [25]:
df = pd.read_csv("augment.csv")

In [48]:
# Define the date range
start_date = "2014-04-12 06:38:03"
end_date = "2018-09-11 17:02:26"

# Check for violations and print them
if not df.iloc[:, 0].is_unique:
    print("First column has duplicate values:")
    print(df[df.iloc[:, 0].duplicated(keep=False)])

if not (df.iloc[:, 1] == 25).all():
    print("Second column has values other than 25:")
    print(df[df.iloc[:, 1] != 25])

if not df.iloc[:, 2].apply(lambda x: isinstance(x, str) and len(x) == 11).all():
    print("Third column has values that are not 11-character strings:")
    print(df[~df.iloc[:, 2].apply(lambda x: isinstance(x, str) and len(x) == 11)])

if not df.iloc[:, 2].is_unique:
    print("Third column has duplicate values:")
    print(df[df.iloc[:, 2].duplicated(keep=False)])

if not df.iloc[:, 3].between(start_date, end_date).all():
    print("Fourth column has dates outside the valid range:")
    print(df[~df.iloc[:, 3].between(start_date, end_date)])

Third column has duplicate values:
     log_id  user_id     video_id           watch_date  subscribed  \
33   100254       25  GYgvliQnCbc  2016-01-27 20:18:22         NaN   
48   100269       25  7voF2YTBeb8  2016-08-14 18:56:01         NaN   
50   100271       25  8V8rtr8aaP0  2018-05-02 14:48:53         NaN   
75   100296       25  wcR815SfWOU  2018-01-27 18:02:55         NaN   
82   100303       25  TohYugHlB8I  2015-11-03 17:14:56         NaN   
83   100304       25  GYgvliQnCbc  2016-03-30 22:07:45         NaN   
105  310020       25  rooEBjZWpDc  2015-07-08 21:01:15         NaN   
132  310047       25  kVAztNx0rHQ  2016-11-18 19:44:05         NaN   
139  310054       25  HxySrSbSY7o  2017-03-24 15:02:41         NaN   
158  310073       25  rooEBjZWpDc  2018-03-15 14:08:19         NaN   
161  310076       25  HxySrSbSY7o  2018-05-10 18:53:41         NaN   
162  310077       25  kVAztNx0rHQ  2018-05-29 20:27:58         NaN   
170  310085       25  kVAztNx0rHQ  2018-09-03 18:01:22 

In [19]:
# Fix third column to ensure uniqueness and 11-character length
existing = set(df.iloc[:, 2])
df.iloc[:, 2] = df.iloc[:, 2].apply(
    lambda x: x if len(x) == 11 and x not in existing else ''.join(random.choices(string.ascii_letters + string.digits, k=11))
)
df.to_csv("augment.csv", index=False)

### Merging the two datasets

In [50]:
# Load the second dataset
df2 = pd.read_csv("../knowledge/clean_youtube_watch_log.csv")

# Merge the two datasets
df_merged = pd.concat([df, df2], ignore_index=True)

# Shuffle the rows
df_merged = df_merged.sample(frac=1).reset_index(drop=True)

# Save the shuffled dataset
df_merged.to_csv("augmented_clean_youtube_watch_log.csv", index=False)

### Regenerate the thumbnails

In [54]:
import requests
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

os.makedirs('../knowledge/thumbnails', exist_ok=True)

def download_thumbnail(video_id):
    file_path = f'../knowledge/thumbnails/{video_id}.jpg'
    if not os.path.exists(file_path):  # Check if the file already exists
        url = f"https://img.youtube.com/vi/{video_id}/default.jpg"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            with open(file_path, 'wb') as f:
                f.write(response.content)

# Execute in parallel to speed up downloading thumbnails
with ThreadPoolExecutor() as ex:
    futures = [ex.submit(download_thumbnail, vid) for vid in df_merged['video_id']]
    for _ in tqdm(as_completed(futures), total=len(futures)):
        pass

  0%|          | 0/1273 [00:00<?, ?it/s]

### Double-check the merged dataset

In [55]:
# Define the date range
start_date = "2014-04-12 06:38:03"
end_date = "2018-09-11 17:02:26"

# Check for violations and print them
if not df_merged.iloc[:, 0].is_unique:
    print("First column has duplicate values:")
    print(df_merged[df_merged.iloc[:, 0].duplicated(keep=False)])

if not (df_merged.iloc[:, 1] == 25).all():
    print("Second column has values other than 25:")
    print(df_merged[df_merged.iloc[:, 1] != 25])

if not df_merged.iloc[:, 3].between(start_date, end_date).all():
    print("Fourth column has dates outside the valid range:")
    print(df_merged[~df_merged.iloc[:, 3].between(start_date, end_date)])

thumb_dir = "../knowledge/thumbnails"
missing = [vid for vid in df_merged["video_id"].astype(str).unique()
            if not os.path.exists(f"{thumb_dir}/{vid}.jpg")]

print(f"Missing thumbnails: {len(missing)}")
if missing:
    print(missing)

Missing thumbnails: 0


## Generating the questions

In [ ]:
class StructuredOutput(BaseModel):
    factual: List[Dict[str, str]]
    cross_modal: List[Dict[str, str]]
    analytical: List[Dict[str, str]]
    conversational: List[Dict[str, str]]

prompt = """
You are an expert evaluator for multimodal agent systems. 
Your goal is to generate specific test queries for a YouTube watch history bias analysis agent.

# Query Categories 
Your goal is to generate 4 queries for each of these categories:
1. **Factual retrieval**: Direct lookup from watch history CSV
2. **Cross-modal retrieval**: Given a title or a thumbnail (assume you have its url), find a similar video in the history
3. **Analytical multi-hop synthesis**: Detect a bias pattern across multiple videos (e.g., political leaning, sensationalism)
4. **Conversational follow-up**: Memory-dependent queries referencing prior responses

# Architectures
Your goal is to generate queries that can effectively benchmark the answering quality of these agent architectures:
1. Vanilla: No KB/retrieval
2. Text RAG: CSV + text-only indexing  
3. MM-RAG: CSV + thumbnails + hybrid indexing
4. Agentic: MM-RAG + bias analysis sub-agent

# Output Format
{
    "factual": [
        { "[question]": "[golden answer]" },
        ...
    ],
    "cross_modal": [
        { "[question]": "[golden answer]" },
        ...
    ],
    "analytical": [
        { "[question]": "[golden answer]" },
        ...
    ],
    "conversational": [
        { "[question]": "[golden answer]" },
        ...
    ]
}

# Rules
- Make queries realistic for YouTube watch history (channels, thumbnails, timestamps)
- Conversational queries MUST reference specific prior query results 
- Cross-modal queries should clearly need thumbnail similarity
- Analytical queries should require pattern detection across multiple videos
- No tool calls - pure query generation only
"""

response = llm \
    .with_structured_output(StructuredOutput, method="json_schema") \
    .invoke(prompt)

with open("results.json", "w") as f:
    f.write(response.model_dump_json()) # type: ignore

In [ ]:
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
import pandas as pd
from pydantic.fields import Field

df = pd.read_csv("augmented_clean_youtube_watch_log.csv")

class QAItem(BaseModel):
    question: str = Field(description="A benchmark question grounded in the dataset.")
    answer: str = Field(description="The golden answer derived from the observed rows in the dataset.")

class StructuredOutput(BaseModel):
    factual: List[QAItem] = Field(description="Exactly 4 factual retrieval questions with golden answers.")
    cross_modal: List[QAItem] = Field(description="Exactly 4 cross-modal retrieval questions with golden answers.")
    analytical: List[QAItem] = Field(description="Exactly 4 analytical multi-hop synthesis questions with golden answers.")
    conversational: List[QAItem] = Field(description="Exactly 4 conversational follow-up questions with golden answers.")

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    final_response: StructuredOutput | None

@tool
def sample_rows(seed: int, n: int = 20) -> str:
    """Return a random sample of rows from the watch-history CSV."""
    sample = df.sample(n=min(n, len(df)), random_state=seed)
    return sample.to_csv(index=False)

tools = [sample_rows]
tool_node = ToolNode(tools)
model_with_tools = llm.bind_tools(tools)

# Main agent node, asks it to analyze the available data and decide wether to call tools again
def call_model(state: AgentState):
    system = """
        You are generating benchmark questions for a YouTube bias-analysis agent.

        You may call tools repeatedly to inspect different random splits of the dataset.
        Before stopping, gather enough evidence to support:
        - 4 factual questions
        - 4 cross-modal questions
        - 4 analytical questions
        - 4 conversational follow-up questions

        You should inspect multiple random samples and look for:
        - exact titles, timestamps, and repeated channels
        - thumbnail/theme patterns for cross-modal similarity
        - detectable bias trends such as political leaning, sensationalism, conspiracy framing
        - candidate question-answer pairs grounded in observed rows

        Do not produce the final JSON yet.
        When you have enough evidence, say clearly that you are ready for final structuring.
    """
    msgs = [HumanMessage(content=system)] + state["messages"]
    resp = model_with_tools.invoke(msgs)
    return {"messages": [resp]}

# Checks wether the model wants to keep using tools
def should_continue(state: AgentState):
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return "structure_response"

# After the model is done, call it one more time with the structured output format
def structure_response(state: AgentState):
    formatter = llm.with_structured_output(StructuredOutput, method="json_schema")
    prompt = """
        You are an expert evaluator for multimodal agent systems.
        Your goal is to generate specific test queries for a YouTube watch history bias analysis agent.

        You have already inspected samples from the dataset using tools. Use ONLY evidence observed in those samples and the tool outputs in the conversation.

        ## Query Categories
        Generate exactly 4 questions for each category:
        1. Factual retrieval: direct lookup from watch history CSV
        2. Cross-modal retrieval: given a title or thumbnail URL, find a similar video in the history
        3. Analytical multi-hop synthesis: detect a bias pattern across multiple videos
        4. Conversational follow-up: memory-dependent queries referencing prior responses

        ## Architectures to benchmark
        Your questions must help differentiate:
        1. Vanilla: no KB / retrieval
        2. Text RAG: CSV + text-only indexing
        3. MM-RAG: CSV + thumbnails + hybrid indexing
        4. Agentic: MM-RAG + bias analysis sub-agent

        ## Rules
        - Return exactly 4 items per category.
        - Every question must have a grounded golden answer.
        - Golden answers must be based on rows actually observed through the tools.
        - Conversational questions must refer to earlier generated questions or answers.
        - Cross-modal questions must clearly require thumbnail/title similarity.
        - Analytical questions must test meaningful bias patterns in the data.
        - Avoid vague or generic questions.
        - Do not invent entities not supported by the observed samples.
    """
    resp = formatter.invoke(state["messages"] + [HumanMessage(content=prompt)])
    return {"final_response": resp}

graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", tool_node)
graph.add_node("structure_response", structure_response)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", "structure_response": "structure_response"})
graph.add_edge("tools", "agent")
graph.add_edge("structure_response", END)

result = graph.compile().invoke({
    "messages": [HumanMessage(content="Create 4 benchmark questions per category grounded in this dataset.")],
    "final_response": None,
})

with open("benchmark.json", "w") as f:
    f.write(result["final_response"].model_dump_json()) # type: ignore